## UCB for Multi-Armed Bandits

In [2]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any
from pprint import pprint as pp

In [24]:
styles = [3, 5, 6]
efforts = ['default']#,"medium","high"]
# styles = [style + '_' + effort for effort in efforts for style in styles]
ucb_path: str = f'../src/optimal_explorer/strategies/bandits/logs/ucb_mab_bernoulli.jsonl'
random_path: str = f'../src/optimal_explorer/strategies/bandits/logs/Random_bernoulli.jsonl'


models = [
    "Gemini Pro 2.5",
    "DeepSeek R1",
    # "Claude Opus 4",
    # "Claude 3.5 Sonnet",
    # "OpenAI o3",
]

model_ids = [
    "google/gemini-2.5-pro",
    "deepseek/deepseek-r1-0528",
    # "anthropic/claude-opus-4",
    # "anthropic/claude-3.5-sonnet",
    # "openai/o3",
]

results_paths: List[str] = [
    f'../src/optimal_explorer/strategies/bandits/logs/game_results/style{style}_{model_id.split("/")[-1]}_default_bernoulli.jsonl'
    for style in styles for model_id in model_ids
]

In [25]:
results_paths

['../src/optimal_explorer/strategies/bandits/logs/game_results/style3_gemini-2.5-pro_default_bernoulli.jsonl',
 '../src/optimal_explorer/strategies/bandits/logs/game_results/style3_deepseek-r1-0528_default_bernoulli.jsonl',
 '../src/optimal_explorer/strategies/bandits/logs/game_results/style5_gemini-2.5-pro_default_bernoulli.jsonl',
 '../src/optimal_explorer/strategies/bandits/logs/game_results/style5_deepseek-r1-0528_default_bernoulli.jsonl',
 '../src/optimal_explorer/strategies/bandits/logs/game_results/style6_gemini-2.5-pro_default_bernoulli.jsonl',
 '../src/optimal_explorer/strategies/bandits/logs/game_results/style6_deepseek-r1-0528_default_bernoulli.jsonl']

In [30]:
results = []

for results_path in results_paths:
    style = results_path.split('_')[2][-1]
    print(style)
    model_id = results_path.split('_')[2]
    with open(results_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            model = data['model']
            regret = data['regret_per_attempt']
            results.append({
                'game_id': data['game_id'],
                'model': model + f' (s={style})',
                'regret': regret,
                'length': len(data['history']),
                'style': style
            })
results_df = pd.DataFrame(results)

3
3
5
5
6
6


In [31]:
len(results_df)

252

In [32]:
ucb = []
random = []

with open(ucb_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        # regret = [sum(data['regret_per_attempt'][:i]) for i in range(data['num_attempts'])]
        regret = data['regret_per_attempt']
        ucb.append({
            'game_id': data['game_id'],
            'model': 'UCB',
            'regret': regret,
            'length': data['num_attempts'],
        })
ucb_df = pd.DataFrame(ucb)

with open(random_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        # regret = [sum(data['regret_per_attempt'][:i]) for i in range(data['num_attempts'])]
        regret = data['regret_per_attempt']
        random.append({
            'game_id': data['game_id'],
            'model': 'Random',
            'regret': regret,
            'length': data['num_attempts'],
        })
random_df = pd.DataFrame(random)

In [36]:
results_df = results_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
ucb_df = ucb_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
random_df = random_df.drop_duplicates(subset=['game_id', 'model'], keep='last')

In [37]:
results_df.head(5)

,game_id,model,regret,length,style
0,10,google/gemini-2.5-pro (s=3),"[0.0, 0.7505686939073445, 0.13767240834047056,...",20,3
1,23,google/gemini-2.5-pro (s=3),"[0.42966471996822486, 0.0, 0.1815028444179072,...",20,3
2,37,google/gemini-2.5-pro (s=3),"[0.0, 0.48039842855289927, 0.7517015739001492,...",20,3
3,47,google/gemini-2.5-pro (s=3),"[0.8609946225428071, 0.0, 0.2457484609353504, ...",20,3
4,42,google/gemini-2.5-pro (s=3),"[0.5761741875625537, 0.0, 0.21872036459851107,...",20,3


In [34]:
ucb_df.head(5)

,game_id,model,regret,length
0,0,UCB,"[0.41484925657370453, 0.2484733941286098, 0.36...",50
1,1,UCB,"[0.3033024887395841, 0.0, 0.7202101186248132, ...",50
2,2,UCB,"[0.18327606420865994, 0.5933447345227724, 0.06...",50
3,3,UCB,"[0.34549518635886256, 0.18814526631533324, 0.6...",50
4,4,UCB,"[0.009244615762565078, 0.4290422056005194, 0.0...",50


In [35]:
random_df.head(5)

,game_id,model,regret,length
0,0,Random,"[0.0, 0.41484925657370453, 0.5802212416752516,...",50
1,1,Random,"[0.7202101186248132, 0.7202101186248132, 0.534...",50
2,2,Random,"[0.18394857373238682, 0.4146223323128212, 0.28...",50
3,3,Random,"[0.7707077784696018, 0.003346134585783367, 0.0...",50
4,4,Random,"[0.7601849591958654, 0.003590094811357414, 0.0...",50


In [41]:
cumulative_regrets = {}
error_bars = {}

for style in styles:
    for mi, model in enumerate(models):
        model_df = results_df[results_df['model'] == model_ids[mi] + f' (s={style})']
        # Convert regret lists to numpy array for easier computation
        regret_array = np.array(model_df['regret'].values.tolist())
    
        # Calculate mean regret per turn
        model_regret = np.mean(regret_array, axis=0)
        cumulative_regret = np.cumsum(model_regret)
        cumulative_regrets[model_ids[mi] + f' (s={style})'] = cumulative_regret
    
        # Calculate standard error of the mean for each turn
        sem = np.std(regret_array, axis=0) / np.sqrt(len(model_df))
        cumulative_sem = np.cumsum(sem)
        error_bars[model_ids[mi] + f' (s={style})'] = cumulative_sem
        print(f'{model} cumulative regret: {cumulative_regret}')

ucb_regret_array = np.array(ucb_df['regret'].values.tolist())
ucb_regret = np.mean(ucb_regret_array, axis=0)
ucb_cumulative_regret = np.cumsum(ucb_regret)
# Calculate standard error of the mean for each turn
sem = np.std(ucb_regret_array, axis=0) / np.sqrt(len(ucb_df))
ucb_cumulative_sem = np.cumsum(sem)

random_regret_array = np.array(random_df['regret'].values.tolist())
random_regret = np.mean(random_regret_array, axis=0)
random_cumulative_regret = np.cumsum(random_regret)
# Calculate standard error of the mean for each turn
sem = np.std(random_regret_array, axis=0) / np.sqrt(len(random_df))
random_cumulative_sem = np.cumsum(sem)

Gemini Pro 2.5 cumulative regret: [0.4140084  0.76511353 1.10691781 1.57160976 1.99092373 2.41453333
 2.8667505  3.29008985 3.69702015 4.11756152 4.5379001  4.915591
 5.30263287 5.58138552 5.89055424 6.13453542 6.36361181 6.5426713
 6.75557819 6.9307419 ]
DeepSeek R1 cumulative regret: [0.4140084  0.76358045 1.12098925 1.58049796 2.01801391 2.40073172
 2.83253308 3.29863381 3.70478205 4.08930361 4.50331201 4.91732041
 5.33132881 5.75355803 6.15323847 6.56236429 6.88310842 7.12143607
 7.3598121  7.51549675]
Gemini Pro 2.5 cumulative regret: [0.4140084  0.76358045 1.12098925 1.57063646 1.99432366 2.40281068
 2.85059031 3.30106177 3.71419502 4.1250215  4.42684107 4.71152163
 4.94036314 5.14886015 5.365289   5.5774152  5.76037444 5.97345074
 6.15413704 6.26477822]
DeepSeek R1 cumulative regret: [0.4140084  0.76814181 1.13740311 1.60678923 2.0344283  2.46009317
 2.91387248 3.34204154 3.77672766 4.20571309 4.48909603 4.7816575
 5.04916704 5.25934238 5.50150307 5.72697405 5.9463764  6.1732442

In [47]:
fig = go.Figure()

colors = [px.colors.qualitative.Dark24[i] for i in [1, 10, 6, 15, 19]]

latex_font = dict(
    family="Latin Modern Roman, Times New Roman, serif",
    size=14,
    color="black"
)

# x_ucb = list(range(len(ucb_cumulative_regret)))
# x_random = list(range(len(random_cumulative_regret)))

x_ucb = list(range(20))
x_random = list(range(20))

# UCB shaded error region
# fig.add_trace(go.Scatter(
#     x=x_ucb + x_ucb[::-1],
#     y=(ucb_cumulative_regret + ucb_cumulative_sem).tolist() + (ucb_cumulative_regret - ucb_cumulative_sem)[::-1].tolist(),
#     fill='toself',
#     fillcolor='rgba(57,106,177,0.18)',  # blue, low alpha
#     line=dict(color='rgba(255,255,255,0)'),
#     hoverinfo="skip",
#     showlegend=False,
#     name='UCB error'
# ))

# UCB main line
fig.add_trace(go.Scatter(
    x=x_ucb,
    y=ucb_cumulative_regret,
    mode='lines+markers',
    name='UCB',
    line=dict(color='blue', width=4),
    marker=dict(size=6, color='blue')
))

# # Random shaded error region
# fig.add_trace(go.Scatter(
#     x=x_random + x_random[::-1],
#     y=(random_cumulative_regret + random_cumulative_sem).tolist() + (random_cumulative_regret - random_cumulative_sem)[::-1].tolist(),
#     fill='toself',
#     fillcolor='rgba(204,37,41,0.18)',  # red, low alpha
#     line=dict(color='rgba(255,255,255,0)'),
#     hoverinfo="skip",
#     showlegend=False,
#     name='Random error'
# ))

# Random main line
fig.add_trace(go.Scatter(
    x=x_random,
    y=random_cumulative_regret,
    mode='lines+markers',
    name='Random',
    line=dict(color='red', width=4),
    marker=dict(size=6, color='red')
))

for style in styles:
    for mi, model in enumerate(models):
        name = model_ids[mi] + f' (s={style})'
        # shaded error region
        # fig.add_trace(go.Scatter(
        #     x=x_ucb + x_ucb[::-1],
        #     y= (cumulative_regrets[name] + error_bars[name]).tolist() + (cumulative_regrets[name] - error_bars[name])[::-1].tolist(),
        #     fill='toself',
        #     # fillcolor=f'rgba{colors[mi][3:-1]},0.18)',  # use mi index in color with low alpha
        #     # line=dict(color='rgba(255,255,255,0)'),
        #     hoverinfo="skip",
        #     showlegend=False,
        #     # name='UCB error'
        # ))

        # main line
        fig.add_trace(go.Scatter(
            x=x_ucb,
            y= cumulative_regrets[name],
            mode='lines+markers',
            name=name,
            line=dict(color=colors[mi], width=4),
            marker=dict(size=6, color='blue')
        ))


fig.update_layout(
    title='',
    xaxis_title='Step (Bernoulli MAB, 10 arms, medium noise)',
    yaxis_title='Cumulative Regret',
    font=latex_font,
    legend=dict(
        title='Strategies',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='black',
        borderwidth=1,
        x=0.01,
        y=0.99,
        xanchor='left',
        yanchor='top'
    ),
    template='plotly_white',
    width=700,
    height=500,
    yaxis=dict(
        title='Cumulative Regret',
        side='right',
        tickmode='linear'
    ),
    xaxis=dict(
        tickmode='linear',
        dtick=10
    )
)
fig.show()

In [71]:
# Use Plotly's Dark24 color set for darker colors
colors = px.colors.qualitative.Dark24[10:]
import random

# Set LaTeX font for all text elements
latex_font = dict(
    family="Latin Modern Roman, Times New Roman, serif",
    size=14,
    color="black"
)

fig = go.Figure()

for si, style in enumerate(styles):
    for mi, model in enumerate(models):
        x_vals = list(range(1, 20))
        y_mean = cumulative_regrets[model_ids[mi] + f' (s={style})']
        y_err = error_bars[model_ids[mi] + f' (s={style})']
        color = random.choice(colors)  # Randomly select a color from the list

        # Add shaded error region (as a filled area)
        fig.add_trace(go.Scatter(
            x=x_vals + x_vals[::-1],
            y=(y_mean + y_err).tolist() + (y_mean - y_err)[::-1].tolist(),
            fill='toself',
            fillcolor=f'rgba{tuple(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)) + (0.18,)}',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False,
            name=f"{model} (s={style})"
        ))

        # Add main mean curve, thicker
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=y_mean,
            mode='lines+markers',
            name=model + f' (s={style})',
            line=dict(width=4, color=color),
            marker=dict(size=6, color=color)
        ))

# Add Bayes Optimal shaded error region (as a filled area)
# fig.add_trace(go.Scatter(
#     x=list(range(1, 20)) + list(range(1, 20))[::-1],
#     y=(ucb_cumulative_regret + ucb_cumulative_sem).tolist() + (ucb_cumulative_regret - ucb_cumulative_sem)[::-1].tolist(),
#     fill='toself',
#     fillcolor=f'rgba(0, 0, 0, 0.5)',
#     line=dict(color='rgba(255,255,255,0)'),
#     hoverinfo="skip",
#     showlegend=False,
#     name='Bayes Optimal'
# ))
# Add Bayes Optimal line
fig.add_trace(go.Scatter(
    x=list(range(1, 20)),
    y=ucb_cumulative_regret,
    mode='markers+lines+lines',
    name='UCB',
    line=dict(width=4, color='rgba(0, 0, 0, 0.8)'),
    marker=dict(size=6, color='rgba(0, 0, 0, 0.8)')
))

# Add y=x baseline as a dashed line
baseline_x = list(range(1, 20))
baseline_y = list(range(1, 20))
fig.add_trace(go.Scatter(
    x=baseline_x,
    y=baseline_y,
    mode='lines',
    name='Baseline',
    line=dict(color='black', width=2, dash='dash'),
    showlegend=True
))

fig.update_layout(
    width=700,
    height=520,
    title=dict(
        text='',
        font=latex_font
    ),
    xaxis_title="Episode (Bernoulli MAB, 10 arms, medium noise)",
    yaxis_title="Cumulative Regret",
    font=latex_font,
    xaxis=dict(
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis=dict(
        # range=[1, 7],
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        side='right',  # default, but we want ticks on both sides
        showticksuffix='all',
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis2=dict(
        overlaying='y',
        side='right',
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    legend=dict(
        title='',
        x=0.03,  # left edge, inside plot
        y=0.97,  # top edge, inside plot
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='black',
        borderwidth=1,
        font=latex_font
    ),
    template='plotly_white'
)

# Add yaxis2 to all traces so ticks show on both sides
for trace in fig.data:
    trace.update(yaxis='y')

# set yrange
fig.update_yaxes(range=[0, 11])